# BirdCLEF 2026 — Perch Submission v1
**Perch ONNX direct logits mapped to BirdCLEF 234 species via taxonomy.csv**
No GPU, no pre-computed spectrograms, < 90 min CPU.


In [ ]:
# [1] Install onnxruntime from local wheel (KAGGLE_NO_INTERNET=1)
import subprocess, sys, os, glob

def _find_dir(pattern):
    direct = f"/kaggle/input/{pattern}"
    if os.path.isdir(direct): return direct
    for root, dirs, files in os.walk("/kaggle/input"):
        if pattern in root: return root
    return direct

WHEEL_DIR = _find_dir("birdclef-perch-models")
print(f"WHEEL_DIR: {WHEEL_DIR}")

try:
    import onnxruntime
    print(f"onnxruntime already installed: {onnxruntime.__version__}")
except ImportError:
    wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "*.whl")))
    if wheels:
        print(f"Installing {os.path.basename(wheels[0])}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", wheels[0]],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        import onnxruntime
        print(f"onnxruntime {onnxruntime.__version__} OK")
    else:
        raise RuntimeError(f"No .whl in {WHEEL_DIR}: {os.listdir(WHEEL_DIR)}")


In [ ]:
# [2] Imports
import os as _os
_os.environ["KAGGLE_NO_INTERNET"] = "1"

import numpy as np
import pandas as pd
import onnxruntime as ort
import librosa
import gc, time
from pathlib import Path


In [ ]:
# [3] Auto-detect paths + configuration
def find_dir(pattern):
    d = f"/kaggle/input/{pattern}"
    if _os.path.isdir(d): return d
    for root, dirs, files in _os.walk("/kaggle/input"):
        if pattern in root: return root
    return d

COMP_DIR  = find_dir("birdclef-2026")
MODEL_DIR = find_dir("birdclef-perch-models")
print(f"COMP_DIR:  {COMP_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")

TEST_DIR        = f"{COMP_DIR}/test_soundscapes"
OUTPUT_PATH     = "/kaggle/working/submission.csv"
TAXONOMY_PATH   = f"{COMP_DIR}/taxonomy.csv"
SAMPLE_SUB_PATH = f"{COMP_DIR}/sample_submission.csv"
PERCH_ONNX      = f"{MODEL_DIR}/perch_v2.onnx"
PERCH_LABELS    = f"{MODEL_DIR}/labels.csv"

SR, DURATION = 32000, 5
SEGMENT_SAMPLES = SR * DURATION  # 160000
BATCH_SIZE = 32

sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opts.intra_op_num_threads = 8
providers = ["CPUExecutionProvider"]


In [ ]:
# [4] Load Perch ONNX + taxonomy + sample_submission
perch_sess = ort.InferenceSession(PERCH_ONNX, sess_opts, providers=providers)
perch_in   = perch_sess.get_inputs()[0].name
perch_out  = [o.name for o in perch_sess.get_outputs()]
print(f"Perch outputs: {perch_out}")

perch_labels_df = pd.read_csv(PERCH_LABELS)
taxonomy_df     = pd.read_csv(TAXONOMY_PATH)
sample_sub      = pd.read_csv(SAMPLE_SUB_PATH)

SPECIES_COLS = [c for c in sample_sub.columns if c != "row_id"]
N_SPECIES = len(SPECIES_COLS)
species_to_idx = {sp: i for i, sp in enumerate(SPECIES_COLS)}

print(f"Perch classes: {len(perch_labels_df)}")
print(f"Taxonomy entries: {len(taxonomy_df)}")
print(f"BirdCLEF species: {N_SPECIES}")
print(f"Sample submission rows: {len(sample_sub)}")


In [ ]:
# [5] Build Perch -> BirdCLEF mapping via taxonomy
# taxonomy: primary_label (e.g. '1161364') -> scientific_name ('Guyalna cuta') -> Perch

# Build lookup dicts from taxonomy
tax_id_to_sci = {}
tax_id_to_common = {}
for _, row in taxonomy_df.iterrows():
    pid = str(row["primary_label"])
    tax_id_to_sci[pid] = str(row.get("scientific_name", "")).lower().strip()
    tax_id_to_common[pid] = str(row.get("common_name", "")).lower().strip()

# Build Perch label list
perch_labels_list = []
for i, row in perch_labels_df.iterrows():
    lbl = str(row.get("label", row.get("scientific_name", ""))).lower().strip()
    perch_labels_list.append(lbl)

# Match
bc_to_perch = {}
perch_to_bc = {}

for bc_sp in SPECIES_COLS:
    bc_idx = species_to_idx[bc_sp]
    sci = tax_id_to_sci.get(bc_sp, bc_sp.lower().replace("_", " ")).strip()
    common = tax_id_to_common.get(bc_sp, "").strip()
    matched = False

    # Strategy 1: exact scientific name
    for pi, pl in enumerate(perch_labels_list):
        if pl == sci:
            bc_to_perch[bc_idx] = pi
            perch_to_bc.setdefault(pi, []).append(bc_idx)
            matched = True; break

    # Strategy 2: genus (first word)
    if not matched and " " in sci:
        genus = sci.split()[0]
        for pi, pl in enumerate(perch_labels_list):
            if pl.startswith(genus + " "):
                bc_to_perch[bc_idx] = pi
                perch_to_bc.setdefault(pi, []).append(bc_idx)
                matched = True; break

    # Strategy 3: exact common name
    if not matched and common:
        for pi, pl in enumerate(perch_labels_list):
            if pl == common:
                bc_to_perch[bc_idx] = pi
                perch_to_bc.setdefault(pi, []).append(bc_idx)
                matched = True; break

    # Strategy 4: substring match (for sonotypes etc.)
    if not matched and common:
        for pi, pl in enumerate(perch_labels_list):
            if common in pl or pl in common:
                bc_to_perch[bc_idx] = pi
                perch_to_bc.setdefault(pi, []).append(bc_idx)
                matched = True; break

matched = len(bc_to_perch)
print(f"Matched: {matched}/{N_SPECIES}")

if matched < N_SPECIES:
    unmatched = [sp for sp in SPECIES_COLS if species_to_idx[sp] not in bc_to_perch]
    print(f"Unmatched ({len(unmatched)}):")
    for sp in unmatched[:20]:
        s = tax_id_to_sci.get(sp, "?")
        c = tax_id_to_common.get(sp, "")
        print(f"  {sp} | sci='{s}' | common='{c}'")


In [ ]:
# [7] Run inference on ALL test soundscapes (skip non-audio files)
AUDIO_EXTENSIONS = {".ogg", ".mp3", ".wav", ".flac", ".m4a", ".opus", ".webm"}

all_files = sorted([
    f for f in _os.listdir(TEST_DIR)
    if _os.path.isfile(_os.path.join(TEST_DIR, f)) and not f.startswith(".")
])
test_files = [
    f for f in all_files
    if _os.path.splitext(f)[1].lower() in AUDIO_EXTENSIONS
]
skipped = [f for f in all_files if f not in test_files]

print(f"All files in test_soundscapes: {len(all_files)}")
if skipped:
    print(f"Skipped non-audio: {skipped}")
print(f"Audio files to process: {len(test_files)}")
if test_files:
    print(f"First 3: {test_files[:3]}")
else:
    print("NOTE: No audio files in notebook env. Real test audio is injected during Kaggle scoring.")

t0 = time.time()
results = []

for idx, fname in enumerate(test_files):
    fpath = _os.path.join(TEST_DIR, fname)
    sid = _os.path.splitext(fname)[0]
    try:
        p = predict_soundscape(fpath)
    except Exception as e:
        print(f"  [ERR] {fname}: {e}")
        p = np.zeros((0, N_SPECIES), dtype=np.float32)

    for seg_i in range(len(p)):
        end_t = (seg_i + 1) * DURATION
        results.append([f"{sid}_{end_t}"] + p[seg_i].tolist())

    if (idx+1) % 10 == 0 or idx < 3:
        ela = time.time() - t0
        eta = ela/(idx+1)*len(test_files) if test_files else 0
        print(f"  [{idx+1:3d}/{len(test_files)}] {fname} ({len(p)} seg) {ela:.0f}s/~{eta:.0f}s")
        gc.collect()

total_time = time.time() - t0
print(f"Inference done: {total_time:.0f}s, {len(results)} segments processed")

In [ ]:
# [8] Build submission.csv
if len(results) == 0:
    # No test audio found in notebook env — this is normal during dev.
    # Kaggle injects real test audio during scoring. Output empty placeholder.
    print("WARNING: No predictions generated (test audio hidden during dev).")
    print("Creating empty submission.csv matching sample_submission structure.")
    sub = sample_sub.copy()
    for col in SPECIES_COLS:
        sub[col] = 0.0
else:
    sub = pd.DataFrame(results, columns=["row_id"] + SPECIES_COLS)
    sub = sub.sort_values("row_id").reset_index(drop=True)
    assert list(sub.columns) == list(sample_sub.columns), "Column mismatch!"

sub.to_csv(OUTPUT_PATH, index=False)

nums = sub[SPECIES_COLS]
active = (nums.max(axis=0) > 0.01).sum() if len(sub) > 0 else 0

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(sub)} | Cols: {len(sub.columns)}")
if len(sub) > 0:
    print(f"Mean prob: {nums.values.mean():.6f}")
    print(f"Max prob:  {nums.values.max():.6f}")
    print(f"Active species (>0.01): {active}/{N_SPECIES}")
    print(f"First row_ids: {sub['row_id'].head(3).tolist()}")
    print(f"Last row_ids:  {sub['row_id'].tail(3).tolist()}")
    for i in range(min(3, len(sub))):
        row = sub.iloc[i]
        top5 = row[SPECIES_COLS].nlargest(5)
        if top5.max() > 0.01:
            print(f"  Row {i} top5: {dict(top5.round(4))}")
else:
    print("Empty submission (placeholder). Will be replaced during scoring.")

In [ ]:
# [8] Build submission.csv
sub = pd.DataFrame(results, columns=["row_id"] + SPECIES_COLS)
sub = sub.sort_values("row_id").reset_index(drop=True)

# Verify column order matches sample_submission
assert list(sub.columns) == list(sample_sub.columns), "Column mismatch!"

sub.to_csv(OUTPUT_PATH, index=False)

nums = sub[SPECIES_COLS]
active = (nums.max(axis=0) > 0.01).sum()

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(sub)} | Cols: {len(sub.columns)}")
print(f"Mean prob: {nums.values.mean():.6f}")
print(f"Max prob:  {nums.values.max():.6f}")
print(f"Active species (>0.01): {active}/{N_SPECIES}")
print(f"First row_ids: {sub['row_id'].head(3).tolist()}")
print(f"Last row_ids:  {sub['row_id'].tail(3).tolist()}")

# Show top predictions per row
for i in range(min(3, len(sub))):
    row = sub.iloc[i]
    top5 = row[SPECIES_COLS].nlargest(5)
    if top5.max() > 0.01:
        print(f"  Row {i} top5: {dict(top5.round(4))}")


In [ ]:
print("\\n" + "="*50)
print("SUBMISSION READY")
print("="*50)
print(f"File: {OUTPUT_PATH}")
print(f"Species matched to Perch: {matched}/{N_SPECIES}")
print(f"Audio files processed: {len(test_files)}")
print(f"Segments inferred: {len(results)}")
if len(results) > 0:
    print(f"Total runtime: {total_time:.0f}s")
print("\\nSubmit this file on the competition page!")